In [ ]:
# Info do banco de dados
jdbc_url = "jdbc:sqlserver://18.209.218.63:4563;databaseName=lh_nautical;encrypt=false;trustServerCertificate=true;"

# Foi criado uma secrets com nome de databases
# O nome do usuario foi colocado usando put-secret e nome da variavel lh_nautical_user
# A senha do usuario foi colocado usando put-secret e nome da variavel lh_nautical_password
connection_properties = {
    "user": dbutils.secrets.get("databases", "lh_nautical_user"),
    "password": dbutils.secrets.get("databases", "lh_nautical_password"),
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

# Info do catalog do desenvolvedor
catalog_dev = dbutils.widgets.get("catalog_dev")

# Query para descobrir todas as tabelas do esquema 'dbo'
query_tabelas = "(select table_name from information_schema.tables where table_schema = 'dbo' and table_type = 'Base table') as tabelas"

# Puxamos as lista de tabelas para um DataFrame e convertemos para uma lista em Python
df_tabelas = spark.read.jdbc(
  url = jdbc_url, 
  table = query_tabelas, 
  properties = connection_properties
)

tabelas = df_tabelas.collect() 
lista_tabelas = [row['table_name'] for row in df_tabelas.collect()]

# Ingestão
for nome_tabela in lista_tabelas:
  print(f"Importanto a tabela: {nome_tabela}...")

  tabela_completa = f"dbo.{nome_tabela}"

  df_temp = spark.read.jdbc(
    url = jdbc_url, 
    table = tabela_completa,
    #column = "id",
    #lowerBound = 1,
    #upperBound = 60000,
    #numPartitions = 4, 
    properties = connection_properties
  )

  df_temp.write.format("delta").mode("append").saveAsTable(f"{catalog_dev}.{nome_tabela}")

print("Ingestão finalizada!")